# VIIRS Flood Timelapse — NOAA AWS Only (Personal Project Version)

Builds a timelapse (GIF/MP4) from NOAA JPSS VIIRS-ABI Flood Day granules, pulled directly from the public
`noaa-jpss` S3 bucket (anonymous access, no credentials, no STAC needed).

This is the pre-gap half of a larger project that eventually bridges to a second, post-gap archive via a STAC
catalog — that part is deliberately **left out for now**. This notebook proves the pipeline on one source first:
fetch → decode → normalize → render with date labels → animate.


In [ ]:
# --- Setup -----------------------------------------------------------------
import os, urllib.request, urllib.parse, xml.etree.ElementTree as ET
from datetime import date, timedelta

import numpy as np
import xarray as xr
import rioxarray as rxr

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image, ImageDraw, ImageFont

DATA_DIR = "data"
FRAMES_DIR = "frames"
OUT_DIR = "output"
for d in (DATA_DIR, FRAMES_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

# AOI (Pantanal bbox, WGS84 lon/lat) — inside the GLB035 tile's coverage.
AOI_BBOX = (-58.0, -20.0, -53.0, -15.0)  # (minx, miny, maxx, maxy)
COMMON_CRS = "EPSG:4326"


## S3 helpers

Defined here, before the manifest, since the manifest builder below needs `list_noaa_day` to check which
dates actually have data.


In [ ]:
NOAA_PREFIX = "JPSS_Blended_Products/VIIRS-ABI-Flood-Day/{yyyy}/{mm}/{dd}/"


def list_noaa_day(yyyy, mm, dd, max_keys=50):
    """List .nc granules for one day in the public noaa-jpss bucket (anonymous, no credentials)."""
    base = "https://noaa-jpss.s3.amazonaws.com/"
    prefix = NOAA_PREFIX.format(yyyy=yyyy, mm=mm, dd=dd)
    query = urllib.parse.urlencode({"list-type": "2", "prefix": prefix, "max-keys": str(max_keys)})
    xml_bytes = urllib.request.urlopen(base + "?" + query).read()
    ns = "{http://s3.amazonaws.com/doc/2006-03-01/}"
    keys = [e.text for e in ET.fromstring(xml_bytes).iter(ns + "Key") if e.text.endswith(".nc")]
    return keys


def download_noaa_granule(key, dest_dir=DATA_DIR):
    base = "https://noaa-jpss.s3.amazonaws.com/"
    url = base + urllib.parse.quote(key)
    local_path = os.path.join(dest_dir, os.path.basename(key))
    if not os.path.exists(local_path):
        urllib.request.urlretrieve(url, local_path)
    return local_path


## Manifest — build over a date range

Walks every day in `START_DATE`..`END_DATE`, keeps only granules matching `TILE` (the tile covering the AOI
above), and skips days with no pass over that tile (normal — not every day images every tile).


In [ ]:
START_DATE = date(2025, 9, 10)
END_DATE = date(2025, 10, 30)
TILE = "GLB035"

NOAA_MANIFEST = []
d = START_DATE
while d <= END_DATE:
    yyyy, mm, dd = f"{d.year:04d}", f"{d.month:02d}", f"{d.day:02d}"
    keys = list_noaa_day(yyyy, mm, dd)
    tile_keys = [k for k in keys if TILE in k]
    if tile_keys:
        filename = tile_keys[0].split("/")[-1]
        NOAA_MANIFEST.append((d.isoformat(), filename))
        print(f"{d}: found {filename}")
    else:
        print(f"{d}: no {TILE} granule found")
    d += timedelta(days=1)

print(f"\n{len(NOAA_MANIFEST)} granules found for manifest")


## Download granules

One HTTPS download per manifest entry — skips files already present in `data/`.


In [ ]:
noaa_local_paths = []
for date_str, filename in NOAA_MANIFEST:
    yyyy, mm, dd = date_str.split("-")
    key = NOAA_PREFIX.format(yyyy=yyyy, mm=mm, dd=dd) + filename
    path = download_noaa_granule(key)
    noaa_local_paths.append((date_str, path))
    print("downloaded:", path)


## Decode `WaterDetection`

Values 0-99 are categorical (land/vegetation/snow/ice/cloud/...). Values 100-200 are water fraction, where
`percent_water = value - 100` (100 → 0%, 200 → 100%). Mask out the categorical band before rendering.


In [ ]:
def load_noaa_water_pct(nc_path):
    ds = xr.open_dataset(nc_path)
    wd = ds["WaterDetection"]
    water_pct = wd.where(wd >= 100) - 100  # 0-100 % water; NaN elsewhere
    return water_pct, ds


def clip_to_aoi(water_pct_da, bbox=AOI_BBOX):
    minx, miny, maxx, maxy = bbox
    lat_name = "lat" if "lat" in water_pct_da.coords else "latitude"
    lon_name = "lon" if "lon" in water_pct_da.coords else "longitude"
    lat_slice = slice(maxy, miny) if water_pct_da[lat_name][0] > water_pct_da[lat_name][-1] else slice(miny, maxy)
    return water_pct_da.sel({lat_name: lat_slice, lon_name: slice(minx, maxx)})


## Render frames (date label)

One shared color ramp (0-100% water) so every frame looks consistent across the whole month.


In [ ]:
CMAP = mcolors.LinearSegmentedColormap.from_list(
    "water_fraction", ["#f5f0e6", "#8ec6ff", "#0b3d91"]  # dry -> partial -> fully flooded
)
NORM = mcolors.Normalize(vmin=0, vmax=100)


def render_frame(water_pct_da, date_str, out_path):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(water_pct_da.values, cmap=CMAP, norm=NORM, origin="upper")
    ax.set_axis_off()
    fig.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

    # Overlay date label with PIL
    img = Image.open(out_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 22)
    except OSError:
        font = ImageFont.load_default()
    draw.rectangle([(10, 10), (230, 45)], fill=(0, 0, 0, 160))
    draw.text((16, 14), date_str, fill="white", font=font)
    img.save(out_path)


frame_paths = []
for date_str, nc_path in noaa_local_paths:
    water_pct, ds = load_noaa_water_pct(nc_path)
    water_pct = clip_to_aoi(water_pct)
    water_pct = water_pct.load()   # pull the actual values into memory before closing the file
    ds.close()                     # release the underlying file handle + memory
    out_path = os.path.join(FRAMES_DIR, f"{date_str}_noaa.png")
    render_frame(water_pct, date_str, out_path)
    frame_paths.append((date_str, out_path))

frame_paths.sort(key=lambda t: t[0])
print(f"{len(frame_paths)} frame(s) rendered")


## Assemble to GIF / MP4

With a month of frames this should now show real motion — the flood extent changing day to day.


In [ ]:
ordered_pngs = [p for _, p in frame_paths]
gif_path = os.path.join(OUT_DIR, "viirs_flood_timelapse_noaa.gif")
mp4_path = os.path.join(OUT_DIR, "viirs_flood_timelapse_noaa.mp4")

try:
    import leafmap
    leafmap.images_to_gif(ordered_pngs, gif_path, fps=2, mp4=True)
except Exception as e:
    print("leafmap path failed, falling back to imageio:", e)
    import imageio.v2 as imageio
    frames = [imageio.imread(p) for p in ordered_pngs]
    imageio.mimsave(gif_path, frames, fps=2)
    try:
        imageio.mimsave(mp4_path, frames, fps=2)
    except Exception as e2:
        print("MP4 export skipped (install imageio-ffmpeg):", e2)

print("Saved:", gif_path)
print("Saved:", mp4_path if os.path.exists(mp4_path) else "(mp4 not generated, see log above)")


## Next steps

- Widen or shift `START_DATE`/`END_DATE` to cover a stronger flood event once you've eyeballed this month's result.
- When you're ready to fold this into the org project, the post-gap (PC Pro) half slots in as its own
  fetch + render step feeding the same `frame_paths` list — nothing here needs to change to support that later.
